In [ ]:
import pandas as pd
import os
import numpy as np

# ==========================================
# 1. Configuration and Constants
# ==========================================

# Behavior Classification Thresholds (Liters/person/day)
# Adjust these if necessary to match your article (e.g., 100/140 or 130/160)
THRESHOLD_ENVIRONMENTALIST = 100.0
THRESHOLD_WASTEFUL = 121.5

# Base Paths (Uncomment the active path)
BASE_PATH = 'E:\\Projetos\\ABM-WP' # Currently active path

def clean_income_value(value):
    """
    Converts income string values to float, handling 'X' (anonymized data) as NaN.
    """
    if isinstance(value, str):
        if value.strip().upper() == 'X':
            return np.nan
        # Standardize decimal separator
        value = value.replace('.', '').replace(',', '.')
    try:
        return float(value)
    except (ValueError, TypeError):
        return np.nan

def main():
    print("--- Starting Advanced Behavior Classification (with Census Data) ---")

    # ==========================================
    # 2. Load and Process Consumer Data
    # ==========================================
    consumers_path = os.path.join(BASE_PATH, 'includes', 'Tabela_consumidores_Itapua_com_setor.csv')
    consumers_df = pd.read_csv(consumers_path, sep=',')
    
    # ==========================================
    # 3. Load and Process Income Data
    # ==========================================
    income_path = os.path.join(BASE_PATH, 'includes','ibge_censo2022', 'Agregados_por_setores_renda_responsavel_BR.csv')
    income_df = pd.read_csv(income_path, sep=';')
    
    # Prepare Income Data
    income_df['CD_SETOR'] = income_df['CD_SETOR'].astype(str)
    income_df = income_df.rename(columns={'V06004': 'VL_RENDA_MEDIA_RESPONSAVEL'})
    income_df['VL_RENDA_MEDIA_RESPONSAVEL'] = income_df['VL_RENDA_MEDIA_RESPONSAVEL'].apply(clean_income_value)
    income_df = income_df[['CD_SETOR', 'VL_RENDA_MEDIA_RESPONSAVEL']]

    # ==========================================
    # 4. Load and Process Consumption History
    # ==========================================
    consumption_path = os.path.join(BASE_PATH, 'includes', 'Tabela_consumo_Itapua_60m.csv')
    consumption_df = pd.read_csv(consumption_path, sep=';')

    # Calculate average consumption of the last 12 months
    consumption_df['AM_REFERENCIA'] = pd.to_datetime(consumption_df['AM_REFERENCIA'], format='%Y%m')
    consumption_df = consumption_df.sort_values(by=['SK_MATRICULA', 'AM_REFERENCIA'], ascending=[True, False])
    
    last_12_months = consumption_df.groupby('SK_MATRICULA').head(12)
    avg_consumption = last_12_months.groupby('SK_MATRICULA')['HCLQTCON'].mean().reset_index()
    avg_consumption.rename(columns={'HCLQTCON': 'NN_MEDIA_CONSUMO'}, inplace=True)

    # Merge Consumption Average
    consumers_df = pd.merge(consumers_df, avg_consumption, on='SK_MATRICULA', how='left')

    # ==========================================
    # 5. Merge Income Data
    # ==========================================
    consumers_df['CD_SETOR_ORIGINAL'] = consumers_df['CD_SETOR'].astype(str)
    consumers_df = pd.merge(consumers_df, income_df, left_on='CD_SETOR_ORIGINAL', right_on='CD_SETOR', how='left', suffixes=('_x', '_income'))
    
    consumers_df = consumers_df.drop(columns=['CD_SETOR_income'])
    consumers_df = consumers_df.rename(columns={'CD_SETOR_x': 'CD_SETOR'})

    # ==========================================
    # 6. Load and Process Census Data (IBGE)
    # ==========================================
    census_path = os.path.join(BASE_PATH, 'includes', 'ibge_censo2022', 'Agregados_por_setores_caracteristicas_domicilio1_BR.csv')
    
    # Robust loading (handling separator variations)
    try:
        census_df = pd.read_csv(census_path, sep=';', dtype={'CD_setor': str})
    except pd.errors.ParserError:
        census_df = pd.read_csv(census_path, sep=',', dtype={'CD_setor': str})

    # Rename IBGE columns
    census_df = census_df.rename(columns={'V00005': 'NN_MEDIA_MORADORES_IBGE', 'V00001': 'NN_MEDIA_DOMICILIOS_IBGE'})
    census_df = census_df[['CD_setor', 'NN_MEDIA_MORADORES_IBGE', 'NN_MEDIA_DOMICILIOS_IBGE']]
    
    census_df['CD_SETOR'] = census_df['CD_setor'].astype(str)
    census_df['NN_MEDIA_MORADORES_IBGE'] = pd.to_numeric(census_df['NN_MEDIA_MORADORES_IBGE'], errors='coerce')
    census_df['NN_MEDIA_DOMICILIOS_IBGE'] = pd.to_numeric(census_df['NN_MEDIA_DOMICILIOS_IBGE'], errors='coerce')

    # Calculate Average Residents per Household in the Sector
    census_df['NN_MEDIA_MORADORES_IBGE'] = round(census_df['NN_MEDIA_MORADORES_IBGE'] / census_df['NN_MEDIA_DOMICILIOS_IBGE'], 1)

    # Merge Census Data
    consumers_df = pd.merge(consumers_df, census_df, left_on='CD_SETOR_ORIGINAL', right_on='CD_SETOR', how='left', suffixes=('_app', '_ibge'))
    
    consumers_df = consumers_df.drop(columns=['CD_SETOR_ibge'])
    consumers_df = consumers_df.rename(columns={'CD_SETOR_app': 'CD_SETOR'})

    # ==========================================
    # 7. Data Cleaning and Imputation
    # ==========================================
    
    # Remove records with missing Sector ID
    consumers_df = consumers_df.dropna(subset=['CD_SETOR'])
    consumers_df = consumers_df[consumers_df['CD_SETOR'].astype(str) != '']

    # Create Simplified Sector ID (SXXX)
    consumers_df['CD_SETOR_SIMPLIFICADO'] = 'S' + consumers_df['CD_SETOR'].astype(str).str[7] + consumers_df['CD_SETOR'].astype(str).str[-2:]

    # --- IMPUTATION LOGIC FOR RESIDENTS ---
    # Create analysis column
    consumers_df['NN_MORADORES_ANALISE'] = consumers_df['NN_MORADORES'].copy()
    
    # Condition: If registered residents == 0, replace with IBGE Sector Average
    condition_replace = (consumers_df['NN_MORADORES'] == 0) & (consumers_df['NN_MEDIA_MORADORES_IBGE'].notna())
    
    consumers_df.loc[condition_replace, 'NN_MORADORES_ANALISE'] = \
        consumers_df.loc[condition_replace, 'NN_MEDIA_MORADORES_IBGE'].round(0)
        
    # If still NaN or 0, fallback to 1 resident
    consumers_df['NN_MORADORES_ANALISE'] = consumers_df['NN_MORADORES_ANALISE'].fillna(1)
    consumers_df.loc[consumers_df['NN_MORADORES_ANALISE'] == 0, 'NN_MORADORES_ANALISE'] = 1
    consumers_df['NN_MORADORES_ANALISE'] = consumers_df['NN_MORADORES_ANALISE'].astype(int)

    # ==========================================
    # 8. Behavior Classification
    # ==========================================
    
    # Calculate Daily Consumption (Liters per person per day)
    # Formula: (Avg m3 * 1000) / Residents / 30.5 days
    consumers_df['NN_CONSUMO_DIARIO'] = (
        (consumers_df['NN_MEDIA_CONSUMO'] * 1000 / consumers_df['NN_MORADORES_ANALISE']) / 30.5
    )
    
    # Classify Profiles
    consumers_df['TP_COMPORTAMENTO'] = consumers_df['NN_CONSUMO_DIARIO'].apply(
        lambda val: 
            'AMBIENTALISTA' if val < THRESHOLD_ENVIRONMENTALIST
            else 'MODERADO' if THRESHOLD_ENVIRONMENTALIST <= val <= THRESHOLD_WASTEFUL 
            else 'PERDULARIO'
    )

    # ==========================================
    # 9. Final Organization and Main Export
    # ==========================================
    
    columns_order = [
        'SK_MATRICULA', 'NM_LOCALIDADE', 'NM_CATEGORIATARIFARIA', 
        'NM_SITUACAO_IMOVEL', 'NN_MORADORES', 'NN_MORADORES_ANALISE', 
        'ST_PISCINA', 'LAT_GEO', 'LONG_GEO', 'X', 'Y', 
        'CD_SETOR_ORIGINAL', 'CD_SETOR_SIMPLIFICADO', 
        'NN_MEDIA_CONSUMO', 'NN_CONSUMO_DIARIO', 'TP_COMPORTAMENTO', 
        'VL_RENDA_MEDIA_RESPONSAVEL', 'NN_MEDIA_MORADORES_IBGE'
    ]
    
    # Ensure columns exist
    for col in columns_order:
        if col not in consumers_df.columns:
            consumers_df[col] = None
    
    consumers_df = consumers_df[columns_order]
    consumers_df = consumers_df.rename(columns={'CD_SETOR_SIMPLIFICADO': 'CD_SETOR'})

    # Save Main File
    main_output_path = os.path.join(BASE_PATH, 'includes', 'Tabela_consumidores_Itapua_com_setor_comportamento_e_renda.csv')
    consumers_df.to_csv(main_output_path, sep=';', index=False)
    print(f"Main dataset saved: {main_output_path}")

    # ==========================================
    # 10. Generate Profile Averages File
    # ==========================================
    print("\n--- Generating Profile Statistics ---")
    
    profile_stats = []
    
    for profile in ['AMBIENTALISTA', 'MODERADO', 'PERDULARIO']:
        df_profile = consumers_df[consumers_df['TP_COMPORTAMENTO'] == profile]
        
        if len(df_profile) > 0:
            avg_consumption = df_profile['NN_CONSUMO_DIARIO'].mean()
            avg_residents = df_profile['NN_MORADORES_ANALISE'].mean()
            avg_income = df_profile['VL_RENDA_MEDIA_RESPONSAVEL'].mean()
            count = len(df_profile)
            percentage = (count / len(consumers_df)) * 100
            
            profile_stats.append({
                'PERFIL': profile,
                'MEDIA_CONSUMO_L_DIA': round(avg_consumption, 2),
                'MEDIA_MORADORES': round(avg_residents, 2),
                'MEDIA_RENDA': round(avg_income, 2),
                'QTD_RESIDENCIAS': count,
                'PERCENTUAL_TOTAL': round(percentage, 2)
            })
            
            print(f"> {profile}: {count} households ({percentage:.1f}%) | Avg Consumption: {avg_consumption:.1f} L/p/d")

    df_stats = pd.DataFrame(profile_stats)
    stats_output_path = os.path.join(BASE_PATH, 'resultados', 'medias_consumo_por_perfil.csv')
    df_stats.to_csv(stats_output_path, sep=';', index=False)
    print(f"Profile averages saved: {stats_output_path}")

    # ==========================================
    # 11. Generate GAMA Detailed Input File
    # ==========================================
    print("\n--- Generating GAMA Input File ---")
    
    # Select specific columns for GAMA simulation
    gama_df = consumers_df[['SK_MATRICULA', 'TP_COMPORTAMENTO', 'NN_MORADORES_ANALISE', 
                            'VL_RENDA_MEDIA_RESPONSAVEL', 'NN_CONSUMO_DIARIO', 
                            'NN_MEDIA_CONSUMO', 'ST_PISCINA', 'X', 'Y']].copy()
    
    # Rename columns to match GAML agent attributes
    gama_df = gama_df.rename(columns={
        'NN_MORADORES_ANALISE': 'NN_MORADORES',
        'NN_CONSUMO_DIARIO': 'NN_CONSUMO_DIARIO_L',
        'NN_MEDIA_CONSUMO': 'NN_MEDIA_CONSUMO_M3',
        'X': 'LATITUDE',
        'Y': 'LONGITUDE'
    })
    
    # Add derived monthly consumption
    gama_df['CONSUMO_MENSAL_M3'] = (gama_df['NN_CONSUMO_DIARIO_L'] * gama_df['NN_MORADORES'] * 30.5) / 1000
    
    gama_output_path = os.path.join(BASE_PATH, 'includes', 'dados_gama_detalhados.csv')
    gama_df.to_csv(gama_output_path, sep=';', index=False)
    print(f"GAMA detailed file saved: {gama_output_path}")

    # ==========================================
    # 12. Final Statistical Summary
    # ==========================================
    print("\n" + "="*60)
    print("FINAL STATISTICAL SUMMARY")
    print("="*60)
    
    # Basic Stats
    count_stats = consumers_df['TP_COMPORTAMENTO'].value_counts()
    pct_stats = consumers_df['TP_COMPORTAMENTO'].value_counts(normalize=True) * 100
    avg_cons_stats = consumers_df.groupby('TP_COMPORTAMENTO')['NN_CONSUMO_DIARIO'].mean().round(2)
    
    summary_df = pd.DataFrame({
        'Count': count_stats,
        'Percentage (%)': pct_stats.round(2),
        'Avg Cons (L/p/d)': avg_cons_stats
    })
    print(summary_df)
    
    # Income Stats
    if 'VL_RENDA_MEDIA_RESPONSAVEL' in consumers_df.columns:
        income_mean = consumers_df.groupby('TP_COMPORTAMENTO')['VL_RENDA_MEDIA_RESPONSAVEL'].mean().round(2)
        income_median = consumers_df.groupby('TP_COMPORTAMENTO')['VL_RENDA_MEDIA_RESPONSAVEL'].median().round(2)
        
        print(f"\n--- Income Statistics (BRL) ---")
        print(pd.DataFrame({'Mean Income': income_mean, 'Median Income': income_median}))

    print("\nProcessing Complete.")

if __name__ == "__main__":
    main()

--- Starting Advanced Behavior Classification (with Census Data) ---


C:\Users\E124796\AppData\Local\Temp\2\ipykernel_12320\1379726575.py:87: DtypeWarning: Columns (1,3,4,5,7,8,9,10,11,13,14,15,16,17,20,21,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51,52,53,54,56,57,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,77,78,84,87) have mixed types. Specify dtype option on import or set low_memory=False.
  census_df = pd.read_csv(census_path, sep=';', dtype={'CD_setor': str})


Main dataset saved: E:\Projetos\ABMS-WP\includes\Tabela_consumidores_Itapua_com_setor_comportamento_e_renda.csv

--- Generating Profile Statistics ---
> AMBIENTALISTA: 8147 households (53.3%) | Avg Consumption: 42.9 L/p/d
> MODERADO: 1313 households (8.6%) | Avg Consumption: 110.2 L/p/d
> PERDULARIO: 5837 households (38.2%) | Avg Consumption: 299.6 L/p/d
Profile averages saved: E:\Projetos\ABMS-WP\resultados\medias_consumo_por_perfil.csv

--- Generating GAMA Input File ---
GAMA detailed file saved: E:\Projetos\ABMS-WP\includes\dados_gama_detalhados.csv

FINAL STATISTICAL SUMMARY
                  Count  Percentage (%)  Avg Cons (L/p/d)
TP_COMPORTAMENTO                                         
AMBIENTALISTA      8147           53.26             42.92
MODERADO           1313            8.58            110.16
PERDULARIO         5837           38.16            299.60

--- Income Statistics (BRL) ---
                  Mean Income  Median Income
TP_COMPORTAMENTO                            
A